# Blanket grid-validation diagnostics

This notebook discovers statistics CSVs in `StandardGrid` and `FinerGrid`, discards the first 2,000 recorded entries, and compares like-for-like Grashof-number runs.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TRANSIENT_ENTRIES = 2_000
CONTROLLED_GR = 1.0e9

# Work when launched from this directory or from the repository root.
VALIDATION_DIR = Path.cwd()
if not (VALIDATION_DIR / "StandardGrid").is_dir():
    VALIDATION_DIR = VALIDATION_DIR / "fusionB" / "validation"
if not (VALIDATION_DIR / "StandardGrid").is_dir():
    raise FileNotFoundError("Could not locate fusionB/validation/StandardGrid")
FIGURES_DIR = VALIDATION_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pattern = re.compile(r"statistics_Gr_([0-9.eE+-]+)(_controlled_)?\.csv$")
records = []
for path in sorted(VALIDATION_DIR.glob("*Grid/statistics_Gr_*.csv")):
    match = pattern.fullmatch(path.name)
    if not match:
        continue
    frame = pd.read_csv(path)
    required = {"step", "time", "Nu", "Kbar"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    records.append({
        "grid": path.parent.name,
        "gr": float(match.group(1)),
        "controlled": bool(match.group(2)),
        "path": path,
        "raw_entries": len(frame),
        "frame": frame.iloc[TRANSIENT_ENTRIES:].copy(),
    })

if not records:
    raise FileNotFoundError(f"No grid-validation CSV files found below {VALIDATION_DIR}")

[(item["grid"], item["gr"], item["controlled"], len(item["frame"])) for item in records]

In [ ]:
# Match the restrained grayscale style used by the original notebook.
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.facecolor": "#F2F2F2",
    "figure.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#222222",
    "axes.titlecolor": "#111111",
    "axes.titleweight": "bold",
    "grid.color": "white",
    "grid.linewidth": 1.0,
    "legend.frameon": False,
    "lines.linewidth": 1.35,
})

grid_styles = {
    "StandardGrid": dict(color="#111111", linestyle="-"),
    "FinerGrid": dict(color="#737373", linestyle="--"),
}

def gr_title(gr):
    exponent = int(round(np.log10(gr)))
    return rf"$Gr = 10^{{{exponent}}}$"

def gr_filename(gr):
    exponent = int(round(np.log10(gr)))
    return f"gr_1e{exponent:02d}"

def save_pdf(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved {path}")

def energy_spectrum(frame):
    clean = frame[["time", "Kbar"]].dropna()
    if len(clean) < 2:
        return None
    time = clean["time"].to_numpy()
    energy = clean["Kbar"].to_numpy()
    dt = np.median(np.diff(time))
    if not np.isfinite(dt) or dt <= 0:
        return None
    fluctuations = energy - energy.mean()
    window = np.hanning(len(fluctuations))
    frequencies = np.fft.rfftfreq(len(fluctuations), d=dt)
    spectrum = np.abs(np.fft.rfft(fluctuations * window)) ** 2
    spectrum /= (1 / dt) * np.sum(window ** 2)
    if len(spectrum) > 2:
        spectrum[1:-1] *= 2
    return frequencies[1:], spectrum[1:]

def plot_comparison(selected, section_title):
    if not selected:
        print(f"{section_title}: no matching CSV files yet.")
        return
    for gr in sorted({item["gr"] for item in selected}):
        group = [item for item in selected if item["gr"] == gr]
        title = gr_title(gr)
        controller = "random_action" if any(item["controlled"] for item in group) else "uncontrolled"
        file_prefix = f"{controller}_{gr_filename(gr)}"

        fig, axes = plt.subplots(2, 1, figsize=(6.0, 5.0), sharex=True,
                                 constrained_layout=True)
        plotted = 0
        for item in group:
            frame = item["frame"]
            if frame.empty:
                continue
            style = grid_styles.get(item["grid"], {})
            axes[0].plot(frame["time"], frame["Nu"],
                         label=item["grid"], **style)
            axes[1].plot(frame["time"], frame["Kbar"],
                         label=item["grid"], **style)
            plotted += 1
        axes[0].set(title=f"Nusselt number — {title}", ylabel=r"$Nu$")
        if np.isclose(gr, 1.0e8):
            axes[0].set_ylim(54, 60)
        axes[1].set(title="Mean kinetic energy", xlabel="Time",
                    ylabel=r"$\overline{K}$")
        for axis in axes:
            axis.margins(x=0.01)
            if plotted:
                axis.legend(loc="best")
            else:
                axis.text(0.5, 0.5, "Waiting for post-transient data",
                          ha="center", va="center", transform=axis.transAxes)
        save_pdf(fig, f"{file_prefix}_nusselt_kinetic_energy.pdf")
        plt.show()

        fig_log, ax_log = plt.subplots(figsize=(4.0, 3.2), constrained_layout=True)
        fig_linear, ax_linear = plt.subplots(figsize=(4.0, 3.2), constrained_layout=True)
        spectra = 0
        for item in group:
            result = energy_spectrum(item["frame"])
            if result is None:
                continue
            frequencies, spectrum = result
            style = grid_styles.get(item["grid"], {})
            ax_log.loglog(frequencies, spectrum, label=item["grid"], **style)
            ax_linear.plot(frequencies, spectrum, label=item["grid"], **style)
            spectra += 1
        ax_log.set(title=f"Mean kinetic-energy spectrum — {title}",
                   xlabel="Frequency", ylabel=r"PSD of $\overline{K}$")
        ax_log.set_xlim(right=0.20)
        ax_linear.set(title=f"Mean kinetic-energy spectrum — {title} (linear scale)",
                      xlabel="Frequency", ylabel=r"PSD of $\overline{K}$")
        ax_linear.set_xlim(0, 0.20)
        ax_linear.set_ylim(bottom=0)
        for axis in (ax_log, ax_linear):
            axis.margins(x=0.01)
            if spectra:
                axis.legend(loc="best")
            else:
                axis.text(0.5, 0.5, "Waiting for post-transient data",
                          ha="center", va="center", transform=axis.transAxes)
        save_pdf(fig_log, f"{file_prefix}_kinetic_energy_spectrum_log.pdf")
        save_pdf(fig_linear, f"{file_prefix}_kinetic_energy_spectrum_linear.pdf")
        plt.show()

In [ ]:
# Electrode-geometry check using the exact shape currently implemented in the UDF.
electrode_x0 = 0.7
electrode_z0 = np.pi/2
electrode_z_half_width = 1.0
electrode_smoothing = 0.2

electrode_x = np.linspace(-1.0, 1.0, 401)
electrode_z = np.linspace(0.0, 3.0, 601)
Z_electrode, X_electrode = np.meshgrid(electrode_z, electrode_x)

x_shape = (
    np.tanh((X_electrode + electrode_x0) / electrode_smoothing)
    - np.tanh((X_electrode - electrode_x0) / electrode_smoothing)
)
z_shape = (
    np.tanh((Z_electrode - electrode_z0 + electrode_z_half_width)
            / electrode_smoothing)
    - np.tanh((Z_electrode - electrode_z0 - electrode_z_half_width)
              / electrode_smoothing)
)
electrode_shape = 0.25 * x_shape * z_shape

fig, ax = plt.subplots(figsize=(6.0, 3.2), constrained_layout=True)
filled = ax.contourf(Z_electrode, X_electrode, electrode_shape,
                     levels=np.linspace(0.0, 1.0, 21), cmap="Greys")
ax.contour(Z_electrode, X_electrode, electrode_shape,
           levels=[0.5], colors="#D55E00", linewidths=1.2)
ax.plot(electrode_z0, 0.0, marker="+", color="#D55E00",
        markersize=9, markeredgewidth=1.5)
ax.vlines([electrode_z0 - electrode_z_half_width,
           electrode_z0 + electrode_z_half_width],
          -electrode_x0, electrode_x0, colors="#333333",
          linestyles=":", linewidth=0.9)
ax.hlines([-electrode_x0, electrode_x0],
          electrode_z0 - electrode_z_half_width,
          electrode_z0 + electrode_z_half_width, colors="#333333",
          linestyles=":", linewidth=0.9)
ax.set(title=rf"Current electrode shape centred at $(x,z)=(0,{electrode_z0:g})$",
       xlabel=rf"$z$", ylabel=rf"$x$",
       xlim=(electrode_z.min(), electrode_z.max()),
       ylim=(electrode_x.min(), electrode_x.max()), aspect="equal")
fig.colorbar(filled, ax=ax, label=rf"Shape amplitude $S(x,z)$")
save_pdf(fig, "electrode_single_shape.pdf")
plt.show()

In [ ]:
# Complete ten-electrode layout on either y-normal wall.
duct_length = 10.0 * np.pi
electrode_centres = (2 * np.arange(10) + 1) * np.pi / 2.0
duct_x = np.linspace(-1.0, 1.0, 301)
duct_z = np.linspace(0.0, duct_length, 2401)
Z_duct, X_duct = np.meshgrid(duct_z, duct_x)

duct_x_shape = (
    np.tanh((X_duct + electrode_x0) / electrode_smoothing)
    - np.tanh((X_duct - electrode_x0) / electrode_smoothing)
)
duct_electrode_shape = np.zeros_like(X_duct)
for centre in electrode_centres:
    duct_z_shape = (
        np.tanh((Z_duct - centre + electrode_z_half_width)
                / electrode_smoothing)
        - np.tanh((Z_duct - centre - electrode_z_half_width)
                  / electrode_smoothing)
    )
    duct_electrode_shape += 0.25 * duct_x_shape * duct_z_shape

fig, ax = plt.subplots(figsize=(12.0, 2.6), constrained_layout=True)
filled = ax.contourf(Z_duct, X_duct, duct_electrode_shape,
                     levels=np.linspace(0.0, 1.0, 21), cmap="Greys")
ax.contour(Z_duct, X_duct, duct_electrode_shape,
           levels=[0.5], colors="#D55E00", linewidths=0.9)
ax.plot(electrode_centres, np.zeros_like(electrode_centres),
        linestyle="none", marker="+", color="#D55E00",
        markersize=7, markeredgewidth=1.2)
for number, centre in enumerate(electrode_centres, start=1):
    ax.text(centre, 0.92, str(number), ha="center", va="top",
            fontsize=8, color="#333333")
ax.set(title=r"Ten-electrode layout over the full $10\pi$ duct",
       xlabel=r"$z$", ylabel=r"$x$",
       xlim=(0.0, duct_length), ylim=(-1.0, 1.0), aspect="equal")
ax.set_xticks(np.arange(11) * np.pi,
              [r"$0$"] + [rf"${k}\pi$" for k in range(1, 11)])
fig.colorbar(filled, ax=ax, label="Combined shape amplitude")
save_pdf(fig, "electrode_ten_layout.pdf")
plt.show()

## Uncontrolled grid validation

In [ ]:
uncontrolled_records = [item for item in records if not item["controlled"]]
plot_comparison(uncontrolled_records, "Uncontrolled grid validation")

## Grid validation for the random-action controller at $Gr = 10^9$

This section is populated only when `_controlled_` CSV files are present. StandardGrid and FinerGrid use the same deterministic random-action sequence.

In [ ]:
controlled_records = [
    item for item in records
    if item["controlled"] and np.isclose(item["gr"], CONTROLLED_GR)
]
plot_comparison(controlled_records,
                f"Random-action controller grid validation at Gr={CONTROLLED_GR:.0e}")

### Controlled StandardGrid–FinerGrid validation statistics

The controlled files are partitioned into the uncontrolled baseline (`200 <= t <= 300`), control-on (`300 < t <= 400`), and post-control relaxation (`t > 400`) phases. The control-on table is the primary grid-validation result; mixing the three phases would obscure it. Confidence intervals use an autocorrelation-corrected effective sample size, and paired diagnostics quantify time-resolved agreement.

In [ ]:
CONTROLLED_VALIDATION_COLUMNS = {
    "Nu": "Nusselt number",
    "Kbar": "Mean kinetic energy",
    "Tbar": "Mean temperature",
    "QwallYMinus": r"Wall flux $y=-1$",
    "QwallYPlus": r"Wall flux $y=+1$",
    "QwallXPlus": r"Wall flux $x=+1$",
}

def effective_sample_size(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n < 3 or np.isclose(values.std(), 0.0):
        return float(max(n, 1))
    centred = values - values.mean()
    nfft = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(centred, n=nfft)
    acov = np.fft.irfft(spectrum * spectrum.conjugate(), n=nfft)[:n]
    acov /= np.arange(n, 0, -1)
    acf = acov / acov[0]
    # Initial-positive-sequence estimate of the integrated autocorrelation time.
    nonpositive = np.flatnonzero(acf[1:] <= 0.0)
    cutoff = int(nonpositive[0] + 1) if len(nonpositive) else n
    tau_int = max(1.0, 1.0 + 2.0 * acf[1:cutoff].sum())
    return float(np.clip(n / tau_int, 1.0, n))

controlled_by_grid = {item["grid"]: item for item in controlled_records}
required_grids = ("StandardGrid", "FinerGrid")
controlled_comparison_status = pd.DataFrame([
    {
        "grid": grid,
        "CSV found": grid in controlled_by_grid,
        "raw entries": controlled_by_grid[grid]["raw_entries"] if grid in controlled_by_grid else 0,
        "post-transient entries": len(controlled_by_grid[grid]["frame"]) if grid in controlled_by_grid else 0,
    }
    for grid in required_grids
])

controlled_grid_statistics = pd.DataFrame()
controlled_grid_comparison = pd.DataFrame()
controlled_grid_response = pd.DataFrame()
if not all(grid in controlled_by_grid and not controlled_by_grid[grid]["frame"].empty
           for grid in required_grids):
    print("Controlled-grid comparison is waiting for post-transient CSV data from both grids.")
    display(controlled_comparison_status)
else:
    prepared = {}
    for grid in required_grids:
        # Reload the complete file because the phase boundaries, rather than
        # the notebook-wide entry cutoff, define this comparison. Kbar is
        # already (1/2V) integral(u.u) dV and must not be doubled.
        frame = pd.read_csv(controlled_by_grid[grid]["path"])
        prepared[grid] = frame.sort_values("time").drop_duplicates("time")

    common_end = min(frame["time"].max() for frame in prepared.values())
    phase_bounds = {
        "uncontrolled baseline": (200.0, min(300.0, common_end), True),
        "control on": (300.0, min(400.0, common_end), False),
    }
    if common_end > 400.0:
        phase_bounds["post-control relaxation"] = (400.0, common_end, False)
    available_columns = [column for column in CONTROLLED_VALIDATION_COLUMNS
                         if all(column in frame.columns for frame in prepared.values())]

    def select_phase(frame, start, end, include_start):
        lower = frame["time"] >= start if include_start else frame["time"] > start
        return frame[lower & (frame["time"] <= end)].copy()

    statistic_rows = []
    comparison_rows = []
    phase_frames = {}
    for phase, (phase_start, phase_end, include_start) in phase_bounds.items():
        common = {grid: select_phase(frame, phase_start, phase_end, include_start)
                  for grid, frame in prepared.items()}
        if any(frame.empty for frame in common.values()):
            continue
        phase_frames[phase] = common
        for grid, frame in common.items():
            for column in available_columns:
                values = frame[column].dropna().to_numpy(dtype=float)
                n = len(values)
                mean = values.mean() if n else np.nan
                std = values.std(ddof=1) if n > 1 else np.nan
                n_eff = effective_sample_size(values) if n else np.nan
                sem = std / np.sqrt(n_eff) if n > 1 else np.nan
                statistic_rows.append({
                    "phase": phase, "grid": grid,
                    "quantity": CONTROLLED_VALIDATION_COLUMNS[column],
                    "samples": n, "effective samples": n_eff,
                    "mean": mean, "std": std,
                    "CV [%]": 100.0 * std / abs(mean) if mean != 0 else np.nan,
                    "95% CI low": mean - 1.96 * sem,
                    "95% CI high": mean + 1.96 * sem,
                })

        # Interpolate onto shared timestamps for time-resolved metrics.
        reference_grid = min(required_grids, key=lambda grid: len(common[grid]))
        paired_time = common[reference_grid]["time"].to_numpy()
        paired = {grid: {column: np.interp(paired_time, frame["time"], frame[column])
                         for column in available_columns}
                  for grid, frame in common.items()}
        for column in available_columns:
            standard = paired["StandardGrid"][column]
            finer = paired["FinerGrid"][column]
            difference = finer - standard
            standard_mean = common["StandardGrid"][column].mean()
            finer_mean = common["FinerGrid"][column].mean()
            scale = abs(standard_mean)
            correlation = np.corrcoef(standard, finer)[0, 1] if (standard.std() > 0 and finer.std() > 0) else np.nan
            comparison_rows.append({
                "phase": phase, "quantity": CONTROLLED_VALIDATION_COLUMNS[column],
                "StandardGrid mean": standard_mean, "FinerGrid mean": finer_mean,
                "Finer - standard": finer_mean - standard_mean,
                "relative mean difference [%]": 100.0 * (finer_mean - standard_mean) / scale if scale else np.nan,
                "standard deviation ratio (finer/standard)": (common["FinerGrid"][column].std() / common["StandardGrid"][column].std()),
                "paired RMSE": np.sqrt(np.mean(difference**2)),
                "normalised RMSE [%]": 100.0 * np.sqrt(np.mean(difference**2)) / scale if scale else np.nan,
                "paired correlation": correlation,
                "common time start": paired_time.min(), "common time end": paired_time.max(),
                "paired samples": len(paired_time),
            })
    controlled_grid_statistics = pd.DataFrame(statistic_rows)

    controlled_grid_comparison = pd.DataFrame(comparison_rows)

    # Report control response relative to each grid's own baseline mean.
    response_rows = []
    if "uncontrolled baseline" in phase_frames and "control on" in phase_frames:
        for grid in required_grids:
            baseline = phase_frames["uncontrolled baseline"][grid]
            controlled = phase_frames["control on"][grid]
            for column in ("Nu", "Tbar", "Kbar"):
                baseline_mean = baseline[column].mean()
                controlled_mean = controlled[column].mean()
                response_rows.append({
                    "grid": grid, "quantity": CONTROLLED_VALIDATION_COLUMNS[column],
                    "baseline mean": baseline_mean, "control-on mean": controlled_mean,
                    "control response [%]": 100.0 * (controlled_mean / baseline_mean - 1.0),
                })
    controlled_grid_response = pd.DataFrame(response_rows)

    for phase in phase_frames:
        print(f"{phase.title()} — per-grid statistics")
        display(controlled_grid_statistics[controlled_grid_statistics["phase"] == phase].style.format(precision=5))
        print(f"{phase.title()} — grid comparison")
        display(controlled_grid_comparison[controlled_grid_comparison["phase"] == phase].style.format(precision=5))
    print("Control-on response relative to each grid's own uncontrolled baseline")
    display(controlled_grid_response.style.format(precision=5))

## Validation means

In [ ]:
# One row per discovered run; empty post-transient runs remain visible.
summary_rows = []
for item in records:
    frame = item["frame"]
    summary_rows.append({
        "grid": item["grid"],
        "Gr": item["gr"],
        "controller": "random action" if item["controlled"] else "uncontrolled",
        "raw entries": item["raw_entries"],
        "post-transient entries": len(frame),
        "mean Nu": frame["Nu"].mean() if not frame.empty else pd.NA,
        "mean kinetic energy": (2 * frame["Kbar"]).mean() if not frame.empty else pd.NA,
        "status": "ready" if not frame.empty else f"waiting for more than {TRANSIENT_ENTRIES:,} entries",
    })

validation_summary = (pd.DataFrame(summary_rows)
                      .sort_values(["controller", "Gr", "grid"])
                      .reset_index(drop=True))
validation_summary